In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [ ]:
phone: Optional[str] = None

In [ ]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'''
Extract a Resume JSON from this text.
Return ONLY JSON, no markdown.

{raw_text}
''',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = f'''
Fix this JSON to match schema.

Errors:
{e}

Original:
{resp.text}
'''

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

In [ ]:
resumes = [
"""
Ravi Kumar
ravi@gmail.com
9876543210

B.Tech Computer Science
IIT Delhi
2023

Skills:
Python, SQL, Machine Learning, AI, Flask, Git

Projects:
Resume Parser
Chatbot

Experience:
1 year internship
""",

"""
Sneha Reddy
sneha@gmail.com

BSc Data Science
Anna University
2024

Skills:
Python, Excel, Power BI, Pandas, NumPy, Tableau

Projects:
Sales Dashboard

Experience:
6 months internship
""",

"""
Arun Pillai
arun@gmail.com
9998887777

MCA
Kerala University
2022

Skills:
Python, Java, C++, SQL, TensorFlow, Deep Learning,
Flask, Docker, Git

Projects:
AI Assistant
Face Recognition

Experience:
1 year software engineer
"""
]

In [ ]:
results = []

for i, r in enumerate(resumes):

    try:
        parsed = extract_resume(r)

        results.append(parsed)

        print(f'\nResume {i+1}:')
        print(f'Name: {parsed.name}')
        print(f'Skills: {len(parsed.skills)}')
        print(f'Experience: {parsed.experience_years} years')

    except Exception as e:

        print(f'\nResume {i+1}: FAILED')
        print(type(e).__name__, str(e)[:200])

if results:

    print('\n=== Full First Result ===')

    print(results[0].model_dump_json(indent=2))

In [ ]:
try:

    bad = extract_resume('')

    print('Unexpected success:', bad.model_dump_json())

except Exception as e:

    print('Caught gracefully:', type(e).__name__)

    print('Message:', str(e)[:200])

In [ ]:
def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:

    # Reject empty input BEFORE calling Gemini
    if not raw_text.strip():
        raise ValueError("Resume text is empty")

    for attempt in range(max_retries + 1):

        try:

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'''
Extract a Resume JSON from this text.
Return ONLY JSON, no markdown.

{raw_text}
''',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = f'''
Fix this JSON to match schema.

Errors:
{e}

Original:
{resp.text}
'''

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

In [ ]:
try:

    bad = extract_resume('')

    print('Unexpected success:', bad.model_dump_json())

except Exception as e:

    print('Caught gracefully:', type(e).__name__)

    print('Message:', str(e))